<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [1]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np

# DataFrame structure used to store and analyze results across multiple runs and configurations.
import pandas as pd

# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt

# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random

# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2

# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod

# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy

# OS is used for file path manipulations and directory creation.
import os

# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

In [2]:
class Solution(ABC):
    """Generic abstract class for an optimization solution."""

    def __init__(self, repr=None):
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [3]:
class VermeerSolution(Solution):
    """A candidate painting made from 100 colored triangles."""

    def __init__(self, target_image, repr=None):
        # Store the original image that the generated triangle image will be compared against.
        self.target_image = target_image

        # Project canvas size: 300x400 pixels.
        self.width = 300
        self.height = 400

        # Project requirement: use 100 triangles.
        self.num_triangles = 100

        # Calls Solution.__init__, which either stores a provided repr or creates a random one.
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates a random genome of 100 small triangles.
        Each row has: [x1, y1, x2, y2, x3, y3, r, g, b]
        """

        # Matrix with one row per triangle and 9 values per triangle.
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # Pick a random center points in the canvas for the triangle
            cx = randint(0, self.width)
            cy = randint(0, self.height)

            # Define an max radius for the triangles
            radius = 40
            # Generate 3 random x-coordinates and 3 random y-coordinates within the range of the defined radius
            # This allows to control the size of the randomly intialized traingles (can start with small triangles)
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            # Generate one random RGB color for the whole triangle.
            r, g, b = [randint(0, 255) for _ in range(3)]
            # Store the triangle's vertices and color in the genome.
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]

        return random_repr

    def render_canvas(self):
        """Converts the 100-triangle genome into an actual image matrix."""

        # Start from a black image with shape: height x width x RGB channels.
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Extract the three triangle vertices in OpenCV's expected format.
            pts = np.array(
                [[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32
            )

            # Reshape points so cv2.fillPoly can draw the triangle.
            pts = pts.reshape((-1, 1, 2))

            # Extract the triangle color.
            # NOTE: The loaded OpenCV image is BGR, but the project logic treats these as channel values consistently.
            color = (int(gene[6]), int(gene[7]), int(gene[8]))

            # Draw a filled triangle on the canvas.
            cv2.fillPoly(canvas, [pts], color)

        return canvas

    def fitness(self):
        """Calculates RMSE between the target image and the generated triangle image.
        Applies a small penalty to overlapping triangles in order to try and resolve the problem with facial features
        """

        # Render this individual's genome into an image.
        generated_image = self.render_canvas()

        # Convert to float before subtracting, avoiding uint8 overflow/underflow errors.
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # Pixel-by-pixel difference between target and generated image.
        diff = target_float - gen_float
        # RMSE = sqrt(mean(square(error))). Lower RMSE means a better image.
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        # The final fitness is the visual error PLUS the geometric penalty
        return rmse

In [4]:
# Path to João's local copy of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
rafa_path = r"..\Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(rafa_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {rafa_path}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(target_image=original_image)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[191   0 147  40 190   0 226 101 150]
 [ 83  92  73 138 108  96 125 166 115]
 [ 74 363 115 362  87 393 110 173  40]
 [133 244 166 295 172 306 146 118 213]
 [274  52 290  27 300  26 221 104   4]
 [226 235 247 169 235 225 131  29  19]
 [198 170 196 148 175 140 103 115 193]
 [217 220 155 220 150 233 227 141  72]
 [141 289 176 315 149 294  84 117 104]
 [221  38 271   0 250  52 167 227 223]
 [ 81  12 120  76 108  23 165 213 157]
 [  0 198  17 204   0 187 103  65 216]
 [242 365 196 359 187 400 113  69  49]
 [282 258 274 231 284 227  11  47 251]
 [ 90  47 116  49 126   4  56  31 211]
 [281 199 254 237 264 183  45 129  42]
 [170 321 186 329 196 312 244 221  42]
 [120 172 139 199  93 175 221  56  51]
 [ 12 296   0 318   8 331 208  24  59]
 [266 161 275 210 256 191 133  17  19]
 [ 67 267 129 258 114 264 124  63  81]
 [173 351 149 369 204 362 229 127  85]
 [197 151 197 142 207 147  90 150 126]
 [243 255 300 242 282 309 185 194 113]
 [153 356 140 337 182 308  86 217   9]
 [216 134 234 157 266  78

In [5]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(gui_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(target_image=original_image)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[ 43 131 106 118  88 165  86  32 235]
 [153 285 209 304 200 301 150  29  59]
 [ 60 301  44 256   9 319 208 163  37]
 [156  15 136  58 143   0  77 112  47]
 [185  72 138  58 141  54 166 211  55]
 [ 58 187   1 122   0 171 135  16  79]
 [118 181 140 113 171 158   5  36 113]
 [109 176 110 147  51 173 113 178 222]
 [272 129 300 133 244 175  80  40 224]
 [220 400 226 400 284 379  55 184 118]
 [300 256 288 316 299 333  34  77 244]
 [292 297 300 279 285 286 114  26 207]
 [233  62 278 100 246 129 198 250   0]
 [217 210 207 215 205 275  21 237  99]
 [ 66 218  38 145  78 149  32  44 213]
 [ 74 239  13 292   3 311 124 145 185]
 [226 378 202 372 207 350 107 159 250]
 [  0 400   0 374   0 380  49 141  89]
 [126 132 117 140 131 137 173 231  76]
 [ 17 312   1 356   0 302 133 121  87]
 [ 81 392  15 334  69 357  69  85  76]
 [137 400 106 396 135 387 213 122 168]
 [ 93 305 157 360 142 326  56 125 132]
 [  0  57  11  81  43  50 166 172 119]
 [208  82 185 125 191 125 185  98  14]
 [  0  79  55  89   0  68

Now to test out if the initial class functions in producing an image and comparing it to the original painting

In [6]:
"""gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'

original_vermeer = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')

my_first_painting = VermeerSolution(target_image=original_vermeer)

score = round(my_first_painting.fitness(), 2)

print(f'The RMSE Score of random painting is: {score}')"""

"gonçalo_path = 'C:\\CIfO\\Project\\data\\girl_pearl_earing.png'\n\noriginal_vermeer = cv2.imread(r'C:\\CIfO\\Project\\data\\girl_pearl_earing.png')\n\nmy_first_painting = VermeerSolution(target_image=original_vermeer)\n\nscore = round(my_first_painting.fitness(), 2)\n\nprint(f'The RMSE Score of random painting is: {score}')"

#### Vemeer 3

In [ ]:
class VermeerGA3:
    """
    GA variant with:
    - Ranking + Tournament Selection
    - Cycle or OX (Two-point) crossover or Uniform crossover
    - Nudge/Replace Mutation with added Scramble + inversion mutation
    - Manhattan distance for fitness sharing (instead of Euclidean)
    """

    def __init__(
        self,
        target_image,
        pop_size=150,
        generations=2000,
        pc=0.85,
        pm=0.03,
        elitism_size=10,
        tournament_size=4,
        selection_method="ranking",  # "tournament" or "ranking"
        crossover_method="cycle",  # "cycle" or "ox" or "uniform"
        mutation_method="swap",  # "swap" or "scramble" or "inversion"
        use_crossover=True,
        use_mutation=True,
        use_fitness_sharing=False,
        variance_threshold=450,
        save_every=None,
        output_dir=None,
    ):

        self.target_image = target_image
        self.pop_size = pop_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.elitism_size = elitism_size

        # Genetic Operators
        self.selection_method = selection_method
        self.tournament_size = tournament_size
        self.crossover_method = crossover_method
        self.mutation_method = mutation_method

        self.use_crossover = use_crossover
        self.use_mutation = use_mutation
        self.use_fitness_sharing = use_fitness_sharing
        self.variance_threshold = variance_threshold
        self.save_every = save_every
        self.output_dir = output_dir

        # Initial random population
        self.population = [
            VermeerSolution(target_image=self.target_image)
            for _ in range(self.pop_size)
        ]

    # ---------- Fitness sharing with Manhattan distance ----------

    def apply_fitness_sharing(self):
        """
        Fitness sharing using Manhattan (L1) distance between individuals.
        Penalizes crowded solutions to avoid premature convergence.
        """
        radius_share = 0.15
        alpha = 1.0

        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])

        # Manhattan distance between all pairs
        diff = flat_reprs[:, np.newaxis, :] - flat_reprs[np.newaxis, :, :]
        dist_matrix = np.sum(np.abs(diff), axis=-1)

        # Normalize distances to [0, 1]
        min_dist = np.min(dist_matrix)
        max_dist = np.max(dist_matrix)
        if max_dist == min_dist:
            norm_dist_matrix = np.zeros_like(dist_matrix)
        else:
            norm_dist_matrix = (dist_matrix - min_dist) / (max_dist - min_dist)

        # Similarity from distance
        similarity_matrix = np.where(
            norm_dist_matrix < radius_share,
            1.0 - (norm_dist_matrix / radius_share) ** alpha,
            0.0,
        )

        niche_counts = np.sum(similarity_matrix, axis=1)

        for idx, ind in enumerate(self.population):
            ind.shared_fitness = ind.fitness_score * niche_counts[idx]

    # ---------- Evaluation ----------

    def evaluate_population(self):
        """Evaluate and sort population by fitness (best first)."""
        for ind in self.population:
            if not hasattr(ind, "fitness_score"):
                ind.fitness_score = ind.fitness()

        if self.use_fitness_sharing:
            self.apply_fitness_sharing()
            self.population.sort(key=lambda ind: ind.shared_fitness)
        else:
            self.population.sort(key=lambda ind: ind.fitness_score)

    # ---------- Selection Methods ----------
    def select_parent(self):
        """Dispatcher between tournament and ranking selection."""
        if self.selection_method == "ranking":
            return self.ranking_selection()
        return self.tournament_selection()

    # ---------- Tournament selection ----------

    def tournament_selection(self):
        """Picks K random individuals and returns the one with the lowest RMSE."""

        # Randomly select tournament_size candidates.
        tournament = sample(self.population, self.tournament_size)

        if self.use_fitness_sharing:
            winner = sorted(tournament, key=lambda ind: ind.shared_fitness)[0]
        else:
            # The candidate with lowest RMSE wins the tournament.
            winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    # ---------- Ranking selection ----------

    def ranking_selection(self):
        """
        Linear ranking selection.
        Population is sorted ascending (best at index 0), so the best
        individual gets the highest rank weight = n, worst gets 1.
        """
        n = len(self.population)
        ranks = np.arange(n, 0, -1)  # [n, n-1, ..., 1]
        probs = ranks / ranks.sum()
        idx = np.random.choice(n, p=probs)
        return self.population[idx]

    # ---------- Crossover Methods ----------

    def crossover(self, parent1, parent2):
        """Dispatcher for crossover methods."""
        if self.crossover_method == "two_point":
            return self.two_point_crossover(parent1, parent2)
        elif self.crossover_method == "cycle":
            return self.cycle_crossover(parent1, parent2)
        # Default to Uniform
        return self.uniform_crossover(parent1, parent2)

    def uniform_crossover(self, parent1, parent2):
        """Uniform crossover: each triangle is inherited from either Parent 1 or Parent 2."""
        # Create a child object (random_representation of a solution(child)); then replace its genome using parent genes.
        child = VermeerSolution(
            target_image=self.target_image,
        )
        # Mask has one 0/1 value per triangle.
        # 1 means take triangle from parent1; 0 means take triangle from parent2.
        mask = np.random.randint(0, 2, size=(child.num_triangles, 1))
        # np.where applies the mask triangle by triangle.
        child.repr = np.where(mask, parent1.repr, parent2.repr).copy()
        return child

    def cycle_crossover(self, parent1, parent2):
        """
        Cycle crossover, adapted.
        Strict CX needs both parents to be permutations of the same elements.
        Here triangles differ between parents, so we build cycles over
        positions: positions are shuffled then split into random-length
        blocks. Each block (cycle) is filled by alternating parents.
        """
        child = VermeerSolution(target_image=self.target_image)
        n = child.num_triangles

        positions = list(range(n))
        np.random.shuffle(positions)

        new_repr = np.empty_like(parent1.repr)
        i = 0
        use_p1 = True
        while i < n:
            cycle_len = randint(1, max(1, n // 4))
            block = positions[i : i + cycle_len]
            source = parent1 if use_p1 else parent2
            for pos in block:
                new_repr[pos] = source.repr[pos]
            use_p1 = not use_p1
            i += cycle_len

        child.repr = new_repr.copy()
        return child

    def two_point_crossover(self, parent1, parent2):
        """
        Order crossover (OX) (two-point crossover), adapted.
        Standard OX copies a segment from parent1 then fills the rest from
        parent2 in order, skipping duplicates. Here triangles are unique
        to each parent so nothing is ever skipped; OX becomes a clean
        two-cut segment crossover, which preserves its spirit.
        """
        child = VermeerSolution(target_image=self.target_image)
        n = child.num_triangles

        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))

        new_repr = parent2.repr.copy()
        new_repr[c1:c2] = parent1.repr[c1:c2]

        child.repr = new_repr
        return child

    # ---------- Mutation ----------

    def mutate(self, child):
        """
        With probability pm, apply either scramble or inversion mutation
        on the triangle order. Note: pm here is per individual (segment op),
        not per triangle as in GA2.
        """
        # 1. Standard Nudge/Replace Mutation (Crucial for exploring shapes/colors!)
        for i in range(child.num_triangles):
            if random() < self.pm:
                mutation_type = random()
                if mutation_type < 0.90:
                    # 90% of mutations are small nudges to preserve useful structures.
                    child.repr[i][0:6] += np.random.randint(-10, 11, 6)  # coordinates
                    child.repr[i][6:9] += np.random.randint(-15, 16, 3)  # color

                    # Keep x-coordinates inside the image width.
                    child.repr[i][0] = np.clip(child.repr[i][0], 0, child.width)
                    child.repr[i][2] = np.clip(child.repr[i][2], 0, child.width)
                    child.repr[i][4] = np.clip(child.repr[i][4], 0, child.width)

                    # Keep y-coordinates inside the image height.
                    child.repr[i][1] = np.clip(child.repr[i][1], 0, child.height)
                    child.repr[i][3] = np.clip(child.repr[i][3], 0, child.height)
                    child.repr[i][5] = np.clip(child.repr[i][5], 0, child.height)

                    # Keep color channels valid.
                    child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)
                    pass
                else:
                    # Full triangle replace
                    child.repr[i] = [
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, 255),
                        randint(0, 255),
                        randint(0, 255),
                    ]
                    pass
        if random() < self.pm:
            if self.mutation_method == "scramble":
                child = self.scramble_mutation(child)
            elif self.mutation_method == "inversion":
                child = self.inversion_mutation(child)

        if hasattr(child, "fitness_score"):
            del child.fitness_score
        return child

    def scramble_mutation(self, child):
        """Shuffle the triangle order inside a random sub-segment."""
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        segment = child.repr[c1:c2].copy()
        np.random.shuffle(segment)
        child.repr[c1:c2] = segment
        return child

    def inversion_mutation(self, child):
        """Reverse the triangle order inside a random sub-segment."""
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        child.repr[c1:c2] = child.repr[c1:c2][::-1]
        return child

    # ---------- Diversity tracking ----------

    def calculate_population_variance(self):
        """Mean genotypic variance across all genes in the population."""
        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])
        gene_variances = np.var(flat_reprs, axis=0)
        return np.mean(gene_variances)

    def maybe_save_generation(self, gen):
        """Optionally save the current best image every N generations."""
        if self.save_every is None or self.output_dir is None:
            return
        if gen % self.save_every != 0:
            return

        import os

        os.makedirs(self.output_dir, exist_ok=True)
        image = self.population[0].render_canvas()
        cv2.imwrite(f"{self.output_dir}/gen_{gen}.png", image)

    # ---------- Main loop ----------

    def run(self):
        """Main evolutionary loop."""
        print(f"Starting Evolution for {self.generations} generations...")

        fitness_history = []
        variance_history = []

        self.evaluate_population()

        for gen in range(self.generations):
            best_current_solution = min(
                self.population, key=lambda ind: ind.fitness_score
            )
            fitness_history.append(best_current_solution.fitness_score)

            if gen % 50 == 0 or gen == self.generations - 1:
                current_variance = self.calculate_population_variance()
                
            variance_history.append(current_variance)

            # Dynamic fitness sharing
            if current_variance < self.variance_threshold:
                if not self.use_fitness_sharing:
                    print(
                        f"\n Variance dropped to {current_variance:.2f} "
                        f"Fitness Sharing Activated at Gen: {gen}"
                    )
                self.use_fitness_sharing = True
                self.apply_fitness_sharing()

            if gen % 50 == 0:
                print(
                    f"Generation {gen} | Best RMSE: "
                    f"{best_current_solution.fitness_score:.2f} | "
                    f"Variance: {current_variance:.2f}"
                )
                self.maybe_save_generation(gen)

            # Elitism
            next_generation = [
                deepcopy(ind) for ind in self.population[: self.elitism_size]
            ]

            # Fill the rest with offspring
            while len(next_generation) < self.pop_size:
                p1 = self.select_parent()

                if self.use_crossover and random() < self.pc:
                    p2 = self.select_parent()
                    child = self.crossover(p1, p2)
                else:
                    child = deepcopy(p1)

                if self.use_mutation:
                    child = self.mutate(child)

                next_generation.append(child)

            self.population = next_generation
            self.evaluate_population()

        self.evaluate_population()
        print(
            f"Evolution Complete! Final RMSE: "
            f"{self.population[0].fitness_score:.2f}"
        )
        return self.population[0], fitness_history, variance_history

In [8]:
target = cv2.imread(r"C:\CIfO\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png")
pop_size = 100
generations = 500
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
save_every = 500
variance_threshold = 550
use_fitness_sharing = False

In [10]:
# 1. Define Base Directory
base_results_dir = r"C:\CIfO\Project\data\results"

# Create base directory if it does not exist
os.makedirs(base_results_dir, exist_ok=True)

# 2. Create New Folder to be used for each run
existing_runs = [
    folder
    for folder in os.listdir(base_results_dir)
    # Folders will be organized by the Nº of Run, so first folder will be the first trial Run then subsequent runs will be 2, 3, 4 ...
    if folder.startswith("Run")
]
# Create list to track the number of runs performed
run_numbers = []

for folder in existing_runs:
    try:
        number = int(folder.replace("Run", ""))
        run_numbers.append(number)
    except:
        pass
    # Looks at the last run performed and adds the next run max(run_numbers, default=0) + 1
next_run_number = max(run_numbers, default=0) + 1

run_folder = os.path.join(base_results_dir, f"Run{next_run_number}")

os.makedirs(run_folder, exist_ok=True)

print(f"Results will be saved in: {run_folder}")

Results will be saved in: C:\CIfO\Project\data\results\Run5


#### Test run of the final class VermeerGA3

In [11]:
ga = VermeerGA3(
    target_image=target,
    pop_size=pop_size,
    generations=generations,
    pc=pc,
    pm=pm,
    elitism_size=elitism_size,
    crossover_method="cycle",  # or "ox"
    use_fitness_sharing=use_fitness_sharing,
    variance_threshold=variance_threshold,
    save_every=save_every,
    output_dir=run_folder,
)

# best_painting, fitness_history, variance_history = ga.run()

In [ ]:
# Baseline Experimment Implementation
exp_name = "Control_Baseline"
notes = "Standard benchmark run (Tournament + Uniform + Scramble). Fitness Sharing turned off for natural variation testing."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "scramble"
n_runs = 30

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = True

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # --- NOVO: Preparar o dicionário de histórico pedido ---
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # --- NOVO: Gravação detalhada de Hiperparâmetros ---
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(fitness_history, color="black", linewidth=1.5)
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[13:13:08] Starting batch: Control_Baseline (30 Runs)
Run 1/30...
Starting Evolution for 2000 generations...
Generation 0 | Best RMSE: 98.71 | Variance: 8869.60
Generation 50 | Best RMSE: 57.27 | Variance: 2047.21
Generation 100 | Best RMSE: 49.96 | Variance: 863.86
Generation 150 | Best RMSE: 45.52 | Variance: 729.35

 Variance dropped to 529.61 Fitness Sharing Activated at Gen: 158
Variance rose to 590.22 Fitness Sharing Deactivated at Gen: 160

 Variance dropped to 536.11 Fitness Sharing Activated at Gen: 162
Variance rose to 552.52 Fitness Sharing Deactivated at Gen: 168

 Variance dropped to 542.04 Fitness Sharing Activated at Gen: 170
Variance rose to 550.70 Fitness Sharing Deactivated at Gen: 171

 Variance dropped to 521.63 Fitness Sharing Activated at Gen: 172
Generation 200 | Best RMSE: 43.29 | Variance: 528.86
Variance rose to 551.45 Fitness Sharing Deactivated at Gen: 203

 Variance dropped to 542.94 Fitness Sharing Activated at Gen: 206
Variance rose to 564.51 Fitness Sha

KeyboardInterrupt: 

In [ ]:
# Implementation Ranking Selection Variant
exp_name = "Selection_Variant_Ranking"
notes = "Selection Isolation: Ranking test to assess whether it prevents premature convergence. Uniform Crossover and Swap Mutation."

# Algorithm Hyperparameters
selection = "ranking"  # <-- Change Made
crossover = "uniform"
mutation = "scramble"
n_runs = 30

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = True

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(fitness_history, color="blue", linewidth=1.5)  # Cor Azul para o Ranking
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[21:40:58] Starting batch: Selection_Variant_Ranking (30 Runs)
Run 1/30...
Starting Evolution for 500 generations...
Generation 0 | Best RMSE: 99.39 | Variance: 8971.97


KeyboardInterrupt: 

In [ ]:
# Genetic Algorithm Crossover Implementation
exp_name = "Crossover_Variant_TwoPoint"
notes = "Crossover Insulation: Testing with Two-Point (OX) to evaluate layer construction. Tournament Selection and Swap Mutation."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "two_point"  # <-- Change Made
mutation = "scramble"
n_runs = 30

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = True

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="green", linewidth=1.5
    )  # Cor Verde para o Two-Point
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[21:40:51] Starting batch: Crossover_Variant_TwoPoint (30 Runs)
Run 1/30...
Starting Evolution for 500 generations...
Generation 0 | Best RMSE: 98.50 | Variance: 8838.24


KeyboardInterrupt: 

In [ ]:
# Implementation Configuration for Mutation Variant: Inversion
exp_name = "Mutation_Variant_Inversion"
notes = "Mutation Isolation: Inversion test to evaluate drastic space exploration jumps. Tournament Selection and Crossover Uniform."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "inversion"  # <-- Change Made
n_runs = 30

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = True

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="red", linewidth=1.5
    )  # Cor Vermelha para o Scramble
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished! Best final RMSE: {np.mean(final_rmses):.2f}"
)

# --- GUARDAR DADOS GLOBAIS EM CSV ---
history_matrix = np.array(all_runs_history)

raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(generations + 1)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

stats_df = pd.DataFrame(
    {
        "Generation": np.arange(generations + 1),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[21:40:41] Starting batch: Mutation_Variant_Scramble (30 Runs)
Run 1/30...
Starting Evolution for 500 generations...
Generation 0 | Best RMSE: 100.34 | Variance: 8802.22


KeyboardInterrupt: 